In [ ]:
import os
import sys
from os import listdir
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import time
import datetime
import pandas as pd
from termcolor import colored, cprint
import numpy as np
import pandas as pd  # Import Pandas and Numpy to create databases
import tensorflow as tf  # Import tensorflow to do tensor manipulations
import seaborn as sns
import matplotlib.pyplot as plt

from abyss.helpers import metrics_viewer, population_viewer
from abyss.models import create_autoencoder
from abyss.glob import glob
from sklearn.model_selection import train_test_split

# Path phenotypes

In [ ]:
PATH_PHENO = glob.PATH_PHENO

In [ ]:
os.listdir(PATH_PHENO)[0]

In [ ]:
os.listdir(PATH_PHENO)

In [ ]:
df = pd.read_csv(PATH_PHENO + f"/phenos.txt",delimiter="\t")

In [ ]:
population_phenos = df[['sub_pop','super_pop']]

In [ ]:
df.columns

In [ ]:
df[['f.eid','f.12144.2.0','f.12144.3.0']]

In [ ]:
df_non_nan_rows = df[['f.eid','f.12144.2.0']].dropna()

In [ ]:
df_non_nan_rows

In [ ]:
path_pheno = PATH_PHENO + "/" +os.listdir(PATH_PHENO)[6]
path_pheno

In [ ]:
pheno_name = path_pheno.split('/')[-1].split('_')[0]
pheno_name

In [ ]:
column_names = ['FID', 'IID', '0', '1', '2', '3']

In [ ]:
pheno = pd.read_csv(path_pheno, delim_whitespace=True, header=None, names=column_names)
pheno

In [ ]:
pheno['3'].unique()

In [ ]:
# Select only the specified columns
pheno_subset = pheno[['FID', 'IID', '3']]

# Rename column '3' to 'phenotype'
pheno_subset.rename(columns={'3': 'phenotype'}, inplace=True)

In [ ]:
pheno_subset

In [ ]:
# Separate the DataFrame into two based on the 'phenotype' column
phenotype_0 = pheno_subset[pheno_subset['phenotype'] == 0]
phenotype_1 = pheno_subset[pheno_subset['phenotype'] == 1]

# Determine the minimum number of cases and controls for balanced sampling
min_samples = min(len(phenotype_0), len(phenotype_1))

# Sample an equal number of cases and controls for train and holdout sets
train_0, holdout_0 = train_test_split(phenotype_0, test_size=0.3, random_state=42)
train_1, holdout_1 = train_test_split(phenotype_1, test_size=0.3, random_state=42)

# Take the same number of cases and controls from each phenotype
pheno_train = pd.concat([train_0.iloc[:min_samples], train_1.iloc[:min_samples]])
pheno_holdout = pd.concat([holdout_0.iloc[:min_samples], holdout_1.iloc[:min_samples]])

# Shuffle the resulting DataFrames to ensure randomness
pheno_train = pheno_train.sample(frac=1, random_state=42).reset_index(drop=True)
pheno_holdout = pheno_holdout.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
pheno_holdout

In [ ]:
pheno_train.to_pickle(PATH_PHENO + "/" + pheno_name +".pkl")
pheno_holdout.to_pickle(PATH_PHENO + "/" + pheno_name +"_holdout.pkl")

# Checking the data

In [ ]:
path_test_data = "../../../data/data/"
pickle_files = [f for f in os.listdir(path_test_data) if f.split(".")[-1] == "pkl"]

In [ ]:
pd.read_pickle(path_test_data+pickle_files[1])

In [ ]:
path_test_data = "../../../data/data/"
fam_files = [f for f in os.listdir(path_test_data) if f.split(".")[-1] == "fam"]
path_pheno = path_test_data + fam_files[1]

In [ ]:
column_names = ['FID', 'IID', '0', '1', '2', '3']

In [ ]:
pheno = pd.read_csv(path_pheno, delim_whitespace=True, header=None, names=column_names)
pheno